<a href="https://colab.research.google.com/github/tremampeloquio/Auditing-a-Streaming-Subscription-Churn-Model-/blob/main/Streaming_Churn_Rate_Notebook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

In [ ]:
train_df = pd.read_csv(r'/Users/rilnex/Downloads/train.csv')
test_df = pd.read_csv(r'/Users/rilnex/Downloads/test.csv')
sample_df = pd.read_csv(r'/Users/rilnex/Downloads/sample_submission.csv')

In [ ]:
train_df['churned'].value_counts()

In [ ]:
train_df

#EDA (Added Code)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

#Set the visual style for the plots
sns.set_theme(style="whitegrid")

# Define our feature categories
categorical_features = [
    'location',
    'subscription_type',
    'payment_plan',
    'customer_service_inquiries' #Treating as categorical since it's discrete integers (0, 1, 2, etc.)
]

continuous_features = [
    'age',
    'weekly_hours'
]

#1. Plot the Categorical Distributions
for col in categorical_features:
    #Check if the column exists to avoid errors
    if col in train_df.columns:
        plt.figure(figsize=(8, 4))
        #Use countplot for categorical/discrete data
        sns.countplot(data=train_df, x=col, palette='viridis')
        plt.title(f'Distribution of {col.replace("_", " ").title()}')
        plt.ylabel('Number of Users')
        plt.xlabel(col.replace("_", " ").title())
        plt.xticks(rotation=45 if train_df[col].nunique() > 5 else 0) # Rotate labels if there are many categories
        plt.tight_layout()
        plt.show()

#2. Plot the Continuous Distributions
for col in continuous_features:
    if col in train_df.columns:
        plt.figure(figsize=(8, 4))
        #Use histplot for continuous data, add a KDE (Kernel Density Estimate) line
        sns.histplot(data=train_df, x=col, kde=True, bins=30, color='#2c728e')
        plt.title(f'Distribution of {col.replace("_", " ").title()}')
        plt.ylabel('Frequency (Number of Users)')
        plt.xlabel(col.replace("_", " ").title())
        plt.tight_layout()
        plt.show()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

#Set visual style
sns.set_theme(style="whitegrid")

#1. Output Profiling: Distribution of Churn
plt.figure(figsize=(6, 4))
ax = sns.countplot(data=train_df, x='churned', palette='viridis')
plt.title('Output Profiling: Distribution of Churn (Target Variable)')
plt.xlabel('Churned (0 = No, 1 = Yes)')
plt.ylabel('Number of Users')

#Add percentages on top of bars
total = len(train_df)
for p in ax.patches:
    percentage = f'{100 * p.get_height() / total:.1f}%'
    x = p.get_x() + p.get_width() / 2
    y = p.get_height()
    ax.annotate(percentage, (x, y), ha='center', va='bottom')

print("\n== Training Set Churn Rates ==")
plt.show()

In [ ]:
#Profile Key Demographic/Categorical Inputs

#2. Input Profiling: Categorical Features
categorical_cols_to_profile = ['customer_service_inquiries', 'subscription_type']

for col in categorical_cols_to_profile:
    if col not in train_df.columns:
        print(f"Skipping '{col}' - column not found in train_df")
        continue
    plt.figure(figsize=(8, 4))
    sns.countplot(data=train_df, x=col, hue='churned', palette='viridis')
    plt.title(f'Churn Distribution by {col}')
    plt.ylabel('Number of Users')
    plt.show()

In [ ]:
#3. Input Profiling: Numerical Features
numeric_cols_to_profile = ['age', 'weekly_hours']

for col in numeric_cols_to_profile:
    #Check if column exists in the dataframe first
    if col in train_df.columns:
        plt.figure(figsize=(8, 4))
        sns.histplot(data=train_df, x=col, hue='churned', kde=True, bins=30, palette='viridis')
        plt.title(f'Distribution of {col} by Churn Status')
        plt.ylabel('Density / Count')
        plt.show()

In [ ]:
# 4. Correlation Matrix of all Numerical Features
plt.figure(figsize=(10, 8))

#Select only numeric columns for correlation to avoid errors
numeric_df = train_df.select_dtypes(include=['int64', 'float64'])

#Drop customer_id as it's just an identifier, not a predictive feature
if 'customer_id' in numeric_df.columns:
    numeric_df = numeric_df.drop(columns=['customer_id'])

#Calculate and plot correlation
correlation_matrix = numeric_df.corr()
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', fmt=".2f", linewidths=0.5)
plt.title('Feature Correlation Matrix')
plt.show()

# Preprocessing

In [ ]:
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

train_df['customer_service_inquiries'] = train_df['customer_service_inquiries'].map({'Low':0,'Medium':1,'High':2})
test_df['customer_service_inquiries'] = test_df['customer_service_inquiries'].map({'Low':0,'Medium':1,'High':2})

num_features = train_df.select_dtypes(['int64','float64']).columns.tolist()
num_features.remove('churned')
num_features.remove('customer_id')
print("Numeric features:", num_features)

cat_features = train_df.select_dtypes('object').columns.tolist()
print("Categorical features:", cat_features)

num_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

cat_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', num_transformer, num_features),
        ('cat', cat_transformer, cat_features)
    ]
)

X = train_df.drop(columns=['customer_id', 'churned'])
y = train_df['churned']
X = preprocessor.fit_transform(X)

print(f"Preprocessed shape: {X.shape}")

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

if hasattr(X_train, 'toarray'):
    X_train = X_train.toarray()
    X_val = X_val.toarray()

print(f"X_train: {X_train.shape}, X_val: {X_val.shape}")
print(f"Churn rate - train: {y_train.mean():.3f}, val: {y_val.mean():.3f}")

In [ ]:
import torch
from torch import nn, optim
from torch.nn import functional as F
from torch.utils.data import DataLoader, TensorDataset

MLP = nn.Sequential(
    nn.Linear(X_train.shape[1],64),
    nn.ReLU(),
    nn.Linear(64,32),
    nn.ReLU(),
    nn.Linear(32,16),
    nn.ReLU(),
    nn.Linear(16,1),
    nn.Sigmoid()
)

epochs = 30
loss_fn = nn.BCELoss()
optimizer = optim.AdamW(MLP.parameters(),lr=3e-4)

X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train.values, dtype=torch.float32).view(-1, 1)

X_val_tensor = torch.tensor(X_val, dtype=torch.float32)
y_val_tensor = torch.tensor(y_val.values, dtype=torch.float32).view(-1, 1)

dataset = TensorDataset(X_train_tensor, y_train_tensor)
dataloader = DataLoader(dataset, batch_size=32, shuffle=True)

for epoch in range(epochs):
    train_loss = 0
    MLP.train()
    num = 0
    for x_batch, y_batch in dataloader:
        optimizer.zero_grad()
        output = MLP(x_batch)
        loss = loss_fn(output, y_batch)
        loss.backward()
        optimizer.step()
        train_loss += loss.item()
        num += 1
    output = MLP(X_val_tensor)
    val_loss = loss_fn(output,y_val_tensor).item()
    print(f'finished epoch {epoch+1}, training loss is {train_loss}, validation loss is {val_loss}')

#Validation (Added Code)

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

#1. Put the model in evaluation mode and get predictions for the validation set
MLP.eval()
with torch.no_grad():
    val_probabilities = MLP(X_val_tensor).squeeze()
    val_predictions = (val_probabilities > 0.5).int()

#Convert tensors back to numpy arrays for sklearn metrics
y_true = y_val_tensor.numpy()
y_pred = val_predictions.numpy()
y_prob = val_probabilities.numpy()

#2. Calculate the metrics
accuracy = accuracy_score(y_true, y_pred)
precision = precision_score(y_true, y_pred)
recall = recall_score(y_true, y_pred)
f1 = f1_score(y_true, y_pred)
auc = roc_auc_score(y_true, y_prob)

#3. Print the results nicely
print("--- Baseline Validation Metrics ---")
print(f"Accuracy:  {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")
print(f"F1-Score:  {f1:.4f}")
print(f"AUC-ROC:   {auc:.4f}")

In [ ]:
Test_X = test_df.drop(columns=['customer_id'])
Test_X = preprocessor.transform(Test_X)  # transform only - never fit on test data
if hasattr(Test_X, 'toarray'):
    Test_X = Test_X.toarray()
Test_Tensor = torch.tensor(Test_X, dtype=torch.float32)
results = MLP(Test_Tensor)

In [ ]:
results = (results > 0.5).int()
print(results)

In [ ]:
submission_file = pd.DataFrame({
    'customer_id': test_df['customer_id'],
    'churned': results.flatten()
})
submission_file.to_csv('submission.csv', index = False)

#Implementation


### Step 0: Get Model Predictions

In [ ]:
import pandas as pd
import numpy as np
import torch
import shap
from sklearn.metrics import precision_score, recall_score, confusion_matrix
import warnings
warnings.filterwarnings('ignore') #Suppress SHAP warnings for a clean output

In [ ]:
#Put the PyTorch model in evaluation mode to get predictions for the validation set
MLP.eval()
with torch.no_grad():
    val_probabilities = MLP(X_val_tensor).squeeze()
    y_pred = (val_probabilities > 0.5).int().numpy()
    y_actual = y_val_tensor.numpy()

#Reconstruct the validation DataFrame with original features for the audit
X_val_df = train_df.loc[y_val.index].copy()
X_val_df['y_pred'] = y_pred
X_val_df['y_actual'] = y_actual

### Step 1: Define All Subgroups

In [ ]:
#A. Age Brackets
X_val_df['age_bracket'] = pd.cut(X_val_df['age'],
                                 bins=[17, 25, 35, 50, 100],
                                 labels=['18-25', '26-35', '36-50', '51+'])

#B. Generations (Requested by peer feedback)
conditions = [
    (X_val_df['age'] >= 18) & (X_val_df['age'] <= 27),
    (X_val_df['age'] >= 28) & (X_val_df['age'] <= 43),
    (X_val_df['age'] >= 44) & (X_val_df['age'] <= 59),
    (X_val_df['age'] >= 60)
]
choices = ['Gen Z', 'Millennials', 'Gen X', 'Boomers']
X_val_df['generation'] = np.select(conditions, choices, default='Unknown')

#C. Intersectional Group (Age Bracket + Location)
X_val_df['intersectional_group'] = X_val_df['age_bracket'].astype(str) + " - " + X_val_df['location'].astype(str)

### Step 2 & 3: Compute Fairness Metrics Function

In [ ]:
def calculate_audit_metrics(df, group_col):
    results = []
    #Filter out any nan combinations from intersectional groups
    groups = [g for g in df[group_col].unique() if pd.notna(g) and g != 'nan - nan']

    for group in sorted(groups):
        subset = df[df[group_col] == group]
        if len(subset) == 0: continue

        y_true = subset['y_actual']
        y_p = subset['y_pred']

        #Confusion Matrix
        tn, fp, fn, tp = confusion_matrix(y_true, y_p, labels=[0,1]).ravel()

        #Core Metrics
        precision = precision_score(y_true, y_p, zero_division=0)
        recall = recall_score(y_true, y_p, zero_division=0)
        fpr = fp / (fp + tn) if (fp + tn) > 0 else 0
        fnr = fn / (fn + tp) if (fn + tp) > 0 else 0

        #Fairness Metrics
        selection_rate = (tp + fp) / len(subset) #Demographic Parity

        results.append({
            'Group': group,
            'Size': len(subset),
            'Precision (Predictive Parity)': round(precision, 3),
            'Recall': round(recall, 3),
            'FPR': round(fpr, 3),
            'FNR (Missed Churners)': round(fnr, 3),
            'Selection Rate (Dem Parity)': round(selection_rate, 3)
        })
    return pd.DataFrame(results)

#RUN THE AUDITS
print("=== 1. AGE BRACKET AUDIT ===")
display(calculate_audit_metrics(X_val_df, 'age_bracket'))

print("\n=== 2. GENERATIONAL AUDIT ===")
display(calculate_audit_metrics(X_val_df, 'generation'))

print("\n=== 3. SUBSCRIPTION TYPE AUDIT ===")
display(calculate_audit_metrics(X_val_df, 'subscription_type'))

print("\n=== 4. INTERSECTIONAL AUDIT (Age + Location) ===")
display(calculate_audit_metrics(X_val_df, 'intersectional_group').head(10)) #Top 10 to save space

### Step 4: Permutation Importance

In [ ]:
print("\n=== 5. PERMUTATION FEATURE IMPORTANCE (PROXY AUDIT) ===")

#1. Calculate Baseline Accuracy
MLP.eval()
with torch.no_grad():
    baseline_probs = MLP(X_val_tensor).squeeze()
    baseline_preds = (baseline_probs > 0.5).int().numpy()
baseline_acc = accuracy_score(y_val_tensor.numpy(), baseline_preds)

importances = []

#2. Shuffle each column one by one and measure the accuracy drop
for i in range(X_val_tensor.shape[1]):
    shuffled_tensor = X_val_tensor.clone()

    #Randomly shuffle the i-th column
    shuffled_tensor[:, i] = shuffled_tensor[torch.randperm(shuffled_tensor.shape[0]), i]

    with torch.no_grad():
        shuffled_probs = MLP(shuffled_tensor).squeeze()
        shuffled_preds = (shuffled_probs > 0.5).int().numpy()

    shuffled_acc = accuracy_score(y_val_tensor.numpy(), shuffled_preds)

    #Importance is how much the accuracy dropped when we ruined this feature
    drop_in_accuracy = baseline_acc - shuffled_acc
    importances.append(drop_in_accuracy)

#3. Reconstruct names and map dummies back to base features
temp_features = train_df.drop(columns=['churned', 'customer_id'], errors='ignore')
base_cols = temp_features.columns.tolist()

#Try standard dummies, then drop_first dummies
temp_dummies = pd.get_dummies(temp_features)
if len(temp_dummies.columns) == X_val_tensor.shape[1]:
    feature_names = temp_dummies.columns.tolist()
else:
    temp_dummies_drop = pd.get_dummies(temp_features, drop_first=True)
    if len(temp_dummies_drop.columns) == X_val_tensor.shape[1]:
        feature_names = temp_dummies_drop.columns.tolist()
    else:
        feature_names = [f"Feature_{i}" for i in range(X_val_tensor.shape[1])]

#4. Group importances back to their original column names
def get_base_feature(feat_name):
    for col in base_cols:
        if feat_name == col or feat_name.startswith(col + '_'):
            return col
    return feat_name

pi_df = pd.DataFrame({'Feature': feature_names, 'Importance': importances})
pi_df['Base_Feature'] = pi_df['Feature'].apply(get_base_feature)

#Sum the importances of dummy variables to get the total importance of the base feature
grouped_pi = pi_df.groupby('Base_Feature')['Importance'].sum().reset_index()
grouped_pi = grouped_pi.sort_values(by='Importance', ascending=False)

#5. Plot the results!
plt.figure(figsize=(10, 6))
sns.barplot(data=grouped_pi, x='Importance', y='Base_Feature', palette='viridis')
plt.title("Permutation Feature Importance (What drives the churn model?)", fontsize=14)
plt.xlabel("Decrease in Accuracy when Feature is Shuffled", fontsize=12)
plt.ylabel("Original Feature", fontsize=12)
plt.tight_layout()
plt.show()

#Step 4(a): SHAP Attempt #1

In [ ]:
import shap
import torch
import numpy as np

if not isinstance(X_val, torch.Tensor):
    X_val_tensor = torch.tensor(X_val.values if hasattr(X_val, 'values') else X_val, dtype=torch.float32)
else:
    X_val_tensor = X_val

background = X_val_tensor[:100]
sample_tensor = X_val_tensor[:500]

explainer = shap.DeepExplainer(MLP, background)
shap_values = explainer.shap_values(sample_tensor)

# DeepExplainer returns shape (n_samples, n_features, n_outputs) for single-output models.
# Squeeze the trailing output dimension so it matches the (n_samples, n_features) data matrix.
shap_values = np.array(shap_values).squeeze(-1)

ohe_cats = preprocessor.named_transformers_['cat'].named_steps['onehot'].get_feature_names_out(cat_features)
feature_names = list(num_features) + list(ohe_cats)

shap.summary_plot(shap_values, sample_tensor.numpy(), feature_names=feature_names)

# Step 4(b): SHAP Attempt #2

In [ ]:
import shap
import torch
import numpy as np

X_background = X_val[:100].copy() if hasattr(X_val, 'copy') else X_val[:100]
X_sample = X_val[:500].copy() if hasattr(X_val, 'copy') else X_val[:500]

def predict_fn(x):
    x_tensor = torch.tensor(x, dtype=torch.float32)
    MLP.eval()
    with torch.no_grad():
        outputs = MLP(x_tensor)
    return outputs.numpy().flatten()  # flatten (n,1) -> (n,) so SHAP gets a 1D output per sample

explainer = shap.KernelExplainer(predict_fn, X_background)
shap_values = explainer.shap_values(X_sample)

# Squeeze in case of extra trailing dimension
shap_values = np.array(shap_values).squeeze()

ohe_cats = preprocessor.named_transformers_['cat'].named_steps['onehot'].get_feature_names_out(cat_features)
feature_names = list(num_features) + list(ohe_cats)

shap.summary_plot(shap_values, X_sample, feature_names=feature_names)

### Step 5: Downstream Dollar Discount

In [ ]:
#1. AGE DOLLAR IMPACT
print("\n=== 6. DOWNSTREAM DOLLAR IMPACT: AGE BRACKET ===")

#Assumption: The retention discount costs PlaylistPro $15 per user offered
discount_value = 5.00

impact_df = X_val_df.groupby('age_bracket').apply(
    lambda x: pd.Series({
        'Total Users': len(x),
        'Discounts Given': x['y_pred'].sum(),
        'Budget Allocated ($)': x['y_pred'].sum() * discount_value
    })
).reset_index()

total_budget = impact_df['Budget Allocated ($)'].sum()
impact_df['% of Total Budget'] = (impact_df['Budget Allocated ($)'] / total_budget * 100).round(1).astype(str) + '%'
display(impact_df)

#2. SUBSCRIPTION TYPE DOLLAR IMPACT
print("\n=== 7. DOWNSTREAM DOLLAR IMPACT: SUBSCRIPTION TYPE ===")

impact_sub_df = X_val_df.groupby('subscription_type').apply(
    lambda x: pd.Series({
        'Total Users': len(x),
        'Discounts Given': x['y_pred'].sum(),
        'Budget Allocated ($)': x['y_pred'].sum() * discount_value
    })
).reset_index()

total_sub_budget = impact_sub_df['Budget Allocated ($)'].sum()
impact_sub_df['% of Total Budget'] = (impact_sub_df['Budget Allocated ($)'] / total_sub_budget * 100).round(1).astype(str) + '%'

display(impact_sub_df)


#3.INTERSECTIONAL DOLLAR IMPACT (Gen + Sub)
print("\n=== 8. DOWNSTREAM DOLLAR IMPACT: GENERATION + SUBSCRIPTION TYPE ===")

#Create the intersectional column just in case it isn't in X_val_df yet
if 'Gen_Sub_Group' not in X_val_df.columns:
    X_val_df['Gen_Sub_Group'] = X_val_df['generation'].astype(str) + " - " + X_val_df['subscription_type'].astype(str)

impact_inter_df = X_val_df.groupby('Gen_Sub_Group').apply(
    lambda x: pd.Series({
        'Total Users': len(x),
        'Discounts Given': x['y_pred'].sum(),
        'Budget Allocated ($)': x['y_pred'].sum() * discount_value
    })
).reset_index()

total_inter_budget = impact_inter_df['Budget Allocated ($)'].sum()
impact_inter_df['% of Total Budget'] = (impact_inter_df['Budget Allocated ($)'] / total_inter_budget * 100).round(1).astype(str) + '%'

#Sort by Budget Allocated to make the business failure obvious right at the top
impact_inter_df = impact_inter_df.sort_values(by='Budget Allocated ($)', ascending=False).reset_index(drop=True)

display(impact_inter_df)



### Different Intersectional Audit (Generation & Subscription)

In [ ]:
from sklearn.metrics import confusion_matrix


print("\n=== 7. INTERSECTIONAL AUDIT (Generation + Subscription Type) ===")

#Create a new combined feature for the intersection
#Adjust 'val_df', 'generation', and 'subscription_type' to match your actual dataframe column names
X_val_df['Gen_Sub_Group'] = X_val_df['generation'].astype(str) + " - " + X_val_df['subscription_type'].astype(str)

audit_results = []

for group_name, group_data in X_val_df.groupby('Gen_Sub_Group'):
    #Adjust 'y_true' and 'y_pred' to match your actual target and prediction arrays/columns
    y_true = group_data['churned'] #Actual churn
    y_pred = group_data['y_pred'] #Model's binary predictions (0 or 1)

    size = len(group_data)

    #Calculate confusion matrix components
    #We use labels=[0, 1] to ensure it always returns 4 values even if a group has only one class
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()

    #Calculate Metrics safely (avoiding division by zero)
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0
    fpr = fp / (fp + tn) if (fp + tn) > 0 else 0
    fnr = fn / (fn + tp) if (fn + tp) > 0 else 0
    selection_rate = (tp + fp) / size if size > 0 else 0

    audit_results.append({
        'Group': group_name,
        'Size': size,
        'Precision (Predictive Parity)': round(precision, 3),
        'Recall': round(recall, 3),
        'FPR': round(fpr, 3),
        'FNR (Missed Churners)': round(fnr, 3),
        'Selection Rate (Dem Parity)': round(selection_rate, 3)
    })

#Convert to DataFrame and sort to find the most penalized group
intersectional_df = pd.DataFrame(audit_results)
intersectional_df = intersectional_df.sort_values(by='FNR (Missed Churners)', ascending=False).reset_index(drop=True)

#Print the cleanly formatted table
print(intersectional_df.to_string())

#Final Plots for Slides/Report

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Subscription-tier FNR from current audit output
categories = ['Premium', 'Family', 'Student', 'Free']
fnr_values = [23.9, 21.5, 10.8, 3.6]

# Pink for high risk, green for low risk
colors = ['#E91E63', '#E91E63', '#1DB954', '#1DB954']

# Dark theme to match presentation style
plt.style.use('dark_background')
fig, ax = plt.subplots(figsize=(10, 6))
fig.patch.set_facecolor('#121212')
ax.set_facecolor('#121212')

# Horizontal bars
y_pos = np.arange(len(categories))
bars = ax.barh(y_pos, fnr_values, color=colors, height=0.6, edgecolor='none')

# Add value labels inside bars
for bar in bars:
    width = bar.get_width()
    ax.text(
        width - 0.5,
        bar.get_y() + bar.get_height() / 2,
        f'{width:.1f}%',
        va='center',
        ha='right',
        color='white',
        fontweight='bold',
        fontsize=12
    )

# Axis formatting
ax.set_yticks(y_pos)
ax.set_yticklabels(categories, fontsize=14, color='white', fontweight='500')
ax.invert_yaxis()
ax.set_xlabel('False Negative Rate (%)', fontsize=12, color='#B3B3B3', labelpad=15)
ax.set_title('Subscription-Tier False Negative Rate', fontsize=14, fontweight='bold', color='white', pad=12)

for spine in ['top', 'right', 'bottom', 'left']:
    ax.spines[spine].set_visible(False)

ax.tick_params(axis='y', length=0, pad=15)
ax.tick_params(axis='x', colors='#B3B3B3')
ax.grid(axis='x', color='#282828', linestyle='--', alpha=0.7)

plt.tight_layout()
plt.show()


In [ ]:
# Set the visual style
sns.set_theme(style="whitegrid")

#CHART 1: The Ethical Failure (FNR by Gen)
gen_metrics = calculate_audit_metrics(X_val_df, 'generation')
fnr_map = dict(zip(gen_metrics['Group'], gen_metrics['FNR (Missed Churners)']))
gen_order = ['Gen X', 'Millennials', 'Gen Z', 'Boomers']
fnr_data = pd.DataFrame({
    'Generation': [g for g in gen_order if g in fnr_map],
    'False Negative Rate (%)': [round(fnr_map[g] * 100, 1) for g in gen_order if g in fnr_map]
})

plt.figure(figsize=(8, 5))
ax1 = sns.barplot(
    data=fnr_data,
    x='Generation',
    y='False Negative Rate (%)',
    hue='Generation',
    dodge=False,
    palette=['#d9534f', '#f0ad4e', '#5bc0de', '#5cb85c']
)
if ax1.legend_ is not None:
    ax1.legend_.remove()
plt.title('The Ethical Failure: Missed Churners (FNR) by Generation', fontsize=14, fontweight='bold', pad=15)
plt.ylabel('False Negative Rate (%)', fontsize=12)
plt.xlabel('')
plt.ylim(0, max(30, fnr_data['False Negative Rate (%)'].max() + 5))

#Add percentages on top
for p in ax1.patches:
    ax1.annotate(f"{p.get_height():.1f}%", (p.get_x() + p.get_width() / 2., p.get_height()),
                 ha='center', va='bottom', fontsize=12, fontweight='bold', color='black', xytext=(0, 5),
                 textcoords='offset points')
plt.tight_layout()
plt.savefig("fnr_by_generation.png", dpi=300)
plt.show()

#CHART 2: The Business Failure (Budget by Tier)
if 'impact_sub_df' not in globals():
    discount_value = 5
    impact_sub_df = X_val_df.groupby('subscription_type').apply(
        lambda x: pd.Series({
            'Total Users': len(x),
            'Discounts Given': x['y_pred'].sum(),
            'Budget Allocated ($)': x['y_pred'].sum() * discount_value
        })
    ).reset_index()
    total_sub_budget = impact_sub_df['Budget Allocated ($)'].sum()
    impact_sub_df['% of Total Budget'] = (impact_sub_df['Budget Allocated ($)'] / total_sub_budget * 100).round(1).astype(str) + '%'

sub_data = impact_sub_df.copy()
sub_data['Budget Allocated (%)'] = sub_data['% of Total Budget'].str.replace('%', '', regex=False).astype(float)
sub_data = sub_data[['subscription_type', 'Budget Allocated (%)']].rename(columns={'subscription_type': 'Subscription Type'})
sub_data = sub_data.set_index('Subscription Type').reindex(['Free', 'Student', 'Family', 'Premium']).reset_index()

plt.figure(figsize=(8, 5))
ax2 = sns.barplot(data=sub_data, x='Subscription Type', y='Budget Allocated (%)',
                  palette=['#d9534f', '#5bc0de', '#5cb85c', '#5cb85c'])
plt.title('The Business Failure: Retention Budget by Subscription Tier', fontsize=14, fontweight='bold', pad=15)
plt.ylabel('% of Total Budget', fontsize=12)
plt.xlabel('')
plt.ylim(0, 50)

#Add percentages on top
for p in ax2.patches:
    ax2.annotate(f"{p.get_height()}%", (p.get_x() + p.get_width() / 2., p.get_height()),
                 ha='center', va='bottom', fontsize=12, fontweight='bold', color='black', xytext=(0, 5),
                 textcoords='offset points')
plt.tight_layout()
plt.savefig("budget_by_subscription.png", dpi=300)
plt.show()

#CHART 3: Compounding Harm (Intersectional)
# We use Top 3 and Bottom 3 groups to keep the presentation slide uncluttered
if 'impact_inter_df' not in globals():
    discount_value = 5
    X_val_df['Gen_Sub_Group'] = X_val_df['generation'].astype(str) + " - " + X_val_df['subscription_type'].astype(str)
    impact_inter_df = X_val_df.groupby('Gen_Sub_Group').apply(
        lambda x: pd.Series({
            'Total Users': len(x),
            'Discounts Given': x['y_pred'].sum(),
            'Budget Allocated ($)': x['y_pred'].sum() * discount_value
        })
    ).reset_index()
    total_inter_budget = impact_inter_df['Budget Allocated ($)'].sum()
    impact_inter_df['% of Total Budget'] = (impact_inter_df['Budget Allocated ($)'] / total_inter_budget * 100).round(1).astype(str) + '%'

inter_plot_df = impact_inter_df.copy()
inter_plot_df['Budget Allocated (%)'] = inter_plot_df['% of Total Budget'].str.replace('%', '', regex=False).astype(float)
inter_plot_df = inter_plot_df.sort_values(by='Budget Allocated (%)', ascending=False).reset_index(drop=True)

top_n = inter_plot_df.head(3).copy()
bottom_n = inter_plot_df.tail(3).copy()
if not top_n.empty:
    top_n.loc[top_n.index[0], 'Gen_Sub_Group'] = top_n.loc[top_n.index[0], 'Gen_Sub_Group'] + '\n(Top)'
if not bottom_n.empty:
    bottom_n.loc[bottom_n.index[-1], 'Gen_Sub_Group'] = bottom_n.loc[bottom_n.index[-1], 'Gen_Sub_Group'] + '\n(Bottom)'

inter_data = pd.concat([top_n, bottom_n], ignore_index=True)
inter_data = inter_data[['Gen_Sub_Group', 'Budget Allocated (%)']].rename(columns={'Gen_Sub_Group': 'Group'})

plt.figure(figsize=(10, 6))
#Horizontal bar chart works best for long labels
ax3 = sns.barplot(data=inter_data, y='Group', x='Budget Allocated (%)',
                  palette=['#5bc0de', '#5bc0de', '#5bc0de', '#d9534f', '#d9534f', '#d9534f'])
plt.title('Compounding Harm: The Highest vs. Lowest Funded Groups', fontsize=14, fontweight='bold', pad=15)
plt.xlabel('% of Total Budget', fontsize=12)
plt.ylabel('')
plt.xlim(0, 18)

#Add percentages on the right of the bars
for p in ax3.patches:
    ax3.annotate(f"{p.get_width()}%", (p.get_width(), p.get_y() + p.get_height() / 2.),
                 ha='left', va='center', fontsize=12, fontweight='bold', color='black', xytext=(5, 0),
                 textcoords='offset points')

plt.tight_layout()
plt.savefig("intersectional_harm.png", dpi=300)
plt.show()